This document contains two functions from [this GitHub repository](https://github.com/ElisaHeinrich/ordered_conformity), modified to:
* Exclude the case `circle = True`
* Include the case `n < 3`

In [1]:
def dispersal(sorted_sample, k):
    """
    Inputs:
        sorted_sample, which is a list of length n with entries that are
           observed variants between 0 and 1, increasing from lowest to highest.
           E.g., if n=4 it could be [0.01,0.1,0.6,0.7].
        k, which is a positive integer 1,2,3,... but no more than n-1.
    Output: a vector of length n with each entry i being the 'dispersal
        measure' for the point that was at entry i in sorted_sample. The
        dispersal measure for a point is lower if it is closer to other points
        and higher if it is farther from other points. There are k 'other points'
        considered for each point.
    """
    k = int(k)
    n = len(sorted_sample)
    results = []
    for i in range(n):
        pt = sorted_sample[i]  # A given point of interest
        subset_min = max(0, i - k)  # Consider k points below it in the list, unless they'd go below 0
        subset_max = min(n - 1, i + k)  # And k points above it, unless that'd be above the max entry
        sample_subset = sorted_sample[subset_min:subset_max + 1]  # Add +1 to include the last entry
        sample_subset.remove(pt) # Do not want the point of interest in here
        dists = [abs(j - pt) for j in sample_subset]  # Distances of other points to pt
        dists.sort()
        dists = dists[:k]  # Take the k smallest distances
        results.append(sum(dists))
    return results

In [2]:
def prob_choose(x, d, k, pr = 9):
    """
    Inputs:
        x: same as sorted_sample in def dispersal.
        d: the conformity coefficient.
            If d = "min" then use a conformity coefficient near the lower bound;
            If d = "max" then use a conformity coefficient near the upper bound. 
        k: same as k in def dispersal.
        pr: precision, used for rounding.
    Output: a list of length n where each entry i corresponds to P(x_i | x). 
    """
    n = len(x)

    # Consider the case n < 3 
    if n == 1:
        return [1] 
    elif n == 2:
        return [0.5, 0.5] 
    
    if type(d) != str:
        if round(d, pr) == 0: 
            return [1 / n for j in x]
    
    dispersals = dispersal(x, k) # Note that x is already sorted 
    if all(round(j,pr) == round(dispersals[0],pr) for j in dispersals): 
        return [1 / n for j in dispersals]
    
    avg_dispersal = sum(dispersals) / len(dispersals)
    possible_lbs = [] # Possible lower bounds
    possible_ubs = [] # Possible upper bounds
    g_list = []
    denom_I_lst = [l for l in dispersals if round(l, pr) > round(avg_dispersal, pr)]
    denom_I = sum(denom_I_lst)
    num_zero_lst = [1 for l in dispersals if round(l, pr) == 0]
    num_zero = sum(num_zero_lst) # Note that sum([]) is 0 
    if num_zero == 0: # None of the dispersals equals 0  
        denom_II_lst = [1/l for l in dispersals if round(l, pr) < round(avg_dispersal, pr)]
        denom_II = sum(denom_II_lst)
    
    for j in dispersals:
        if round(j, pr) == round(avg_dispersal, pr): # Equal to average dispersal 
            g_i = 0            
        elif round(j, pr) > round(avg_dispersal, pr): # Above average dispersal 
            g_i = -j/denom_I
        elif round(j, pr) < round(avg_dispersal, pr): # Below average dispersal
            if num_zero > 0:
                if round(j, pr) == 0:
                    g_i = 1/num_zero
                else: 
                    g_i = 0
            elif num_zero == 0:
                g_i = (1/j)/denom_II 

        try:
            g_list.append(g_i)
        except:
            print(j, avg_dispersal)
        if g_i > 0:
            possible_lbs.append(-1/g_i)
            possible_ubs.append((n-1)/g_i)
        elif g_i < 0:
            possible_lbs.append((n-1)/g_i)
            possible_ubs.append(-1/g_i)
            
    if d == "min": d = max(possible_lbs) + 1e-5 
    elif d == "max": d = min(possible_ubs) - 1e-5
    return [1/n + g*d/n for g in g_list]